# Feature Engineering 

This notebook starts from the output of the preprocessing pipeline. It creates business-focused lifecycle, profitability, marketing, supply, market, customer-mix, peer-benchmark, momentum and interaction features for predicting product risk one month ahead.

## Notebook overview

1. Load preprocessed outputs and confirm their schema and grain
2. Helper functions 
3. Create lifecycle and profitability features
4. Create campaign and digital-demand features
5. Create supply-chain and return-pressure features
6. Create market-context changes 
7. Compare products with category-region peers
8. Create customer-mix features from the customer-product-month output
9. Add historical momentum, stability and interaction features
10. Audit leakage and select model features
11. Split by target month and fit preprocessing on training data 
12. Export datasets, matrices, IDs, feature documentation and QA metadata

## Leakage policy

The prediction is made after the current month closes. Therefore current-month commercial and market values are allowed. The following are excluded from model features:

- target `next_month_risk_label`;
- `target_month` and entity IDs;
- any column beginning with `future_`;
- audit dates and raw calendar keys;
- preprocessing parameters learned outside the training period.

Rolling features use `.shift(1)` whenever they describe historical baselines, so the current observation is not included in its own history.


# 1. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 190)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

RANDOM_SEED = 42
TARGET_COL = "next_month_risk_label"
KEY_COLS = ["product_id", "region_id", "year_month"]

INPUT_DIR = Path("../data/full_data/preprocessed_data")

OUTPUT_DIR = Path("../data/full_data/feature_engineering")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input folder:  {INPUT_DIR}")
print(f"Output folder: {OUTPUT_DIR}")

Input folder:  ..\data\full_data\preprocessed_data
Output folder: ..\data\full_data\feature_engineering


# 2. Load preprocessing outputs

Two files are required:

- `ews_modeling_dataset_enriched.csv`: one row per product-region-month;
- `customer_product_monthly.csv`: customer composition for each active product-month.

In [2]:
INPUT_FILES = {
    "modeling": "ews_modeling_dataset_enriched.csv",
    "customer_product_monthly": "customer_product_monthly.csv"}
missing_files = [
    filename
    for filename in INPUT_FILES.values()
    if not (INPUT_DIR / filename).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing preprocessing outputs in {INPUT_DIR}: {missing_files}")
modeling_input = pd.read_csv(
    INPUT_DIR / INPUT_FILES["modeling"], low_memory=False)
customer_product_monthly = pd.read_csv(
    INPUT_DIR / INPUT_FILES["customer_product_monthly"], low_memory=False)

input_summary = pd.DataFrame(
    [
        {
            "dataset": "modeling_input",
            "rows": len(modeling_input),
            "columns": modeling_input.shape[1],
            "duplicate_rows": int(modeling_input.duplicated().sum()),
            "missing_cells_pct": round(
                modeling_input.isna().mean().mean() * 100, 2)},
        {
            "dataset": "customer_product_monthly",
            "rows": len(customer_product_monthly),
            "columns": customer_product_monthly.shape[1],
            "duplicate_rows": int(
                customer_product_monthly.duplicated().sum()
            ),
            "missing_cells_pct": round(
                customer_product_monthly.isna().mean().mean() * 100, 2)}])
display(input_summary)


,dataset,rows,columns,duplicate_rows,missing_cells_pct
0,modeling_input,120000,131,0,6.270
1,customer_product_monthly,118882,12,0,0.070


## 2.1 Standardize keys and dates

IDs are stored as strings to prevent a mix of integer, float-like and text IDs during joins. Month fields are normalized to month-start.


In [3]:
def normalize_id(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace(
            {
                "<NA>": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "": pd.NA}))

def to_month(series: pd.Series) -> pd.Series:
    return (
        pd.to_datetime(series, errors="coerce", format="mixed")
        .dt.to_period("M")
        .dt.to_timestamp())

df = modeling_input.copy()
df["product_id"] = normalize_id(df["product_id"])
df["region_id"] = normalize_id(df["region_id"])
df["year_month"] = to_month(df["year_month"])
df["target_month"] = (
    to_month(df["target_month"])
    if "target_month" in df.columns
    else df["year_month"] + pd.offsets.MonthBegin(1))

for date_column in [
    "launch_date", "effective_launch_date", "observed_first_sale_date"
]:
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(
            df[date_column], errors="coerce", format="mixed")

customer_product_monthly["product_id"] = normalize_id(
    customer_product_monthly["product_id"])
customer_product_monthly["region_id"] = normalize_id(
    customer_product_monthly["region_id"])
customer_product_monthly["customer_id"] = normalize_id(
    customer_product_monthly["customer_id"])
customer_product_monthly["year_month"] = to_month(
    customer_product_monthly["year_month"])

display(df[KEY_COLS + ["target_month", TARGET_COL]].head())


,product_id,region_id,year_month,target_month,next_month_risk_label
0,1000,1,2021-01-01,2021-02-01,0.000
1,1000,1,2021-02-01,2021-03-01,0.000
2,1000,1,2021-03-01,2021-04-01,0.000
3,1000,1,2021-04-01,2021-05-01,1.000
4,1000,1,2021-05-01,2021-06-01,0.000


## 2.2 Validate input and observation grain

Feature engineering should fail early if preprocessing no longer supplies a required field. The natural key must remain unique before and after every transformation.


In [4]:
REQUIRED_INPUT_COLUMNS = [
    # IDs, timing, target, and core sales
    "product_id", "region_id", "region", "year_month", "target_month",
    TARGET_COL, "units_sold", "revenue", "unique_customers",
    "avg_discount_pct", "order_count", "gross_profit_eur",
    "gross_margin_pct", "cogs_eur",
    # Existing historical features
    "units_lag_1m", "units_lag_3m", "revenue_lag_1m",
    "rolling_units_mean_3m", "rolling_units_mean_6m",
    "trend_slope_3m", "trend_slope_6m",
    # Product and financial context
    "product_line", "product_category", "lifecycle_stage",
    "effective_launch_date", "base_unit_cost_eur",
    "target_margin_pct",
    # Supply and return context
    "opening_stock_units", "production_units", "ending_stock_units",
    "stockout_flag", "zero_stock_flag", "backorder_units",
    "supply_pressure_index", "return_units", "return_value_eur",
    # Marketing and market context
    "campaign_spend_eur", "website_visits", "product_page_views",
    "demo_requests", "marketing_qualified_leads",
    "market_demand_index", "competitor_pressure_index",
    "seasonality_index", "macro_business_index",
    "market_opportunity_score"]

missing_required_columns = [
    column for column in REQUIRED_INPUT_COLUMNS if column not in df.columns]
if missing_required_columns:
    raise ValueError(
        f"Preprocessing output is missing required columns: "
        f"{missing_required_columns}")

input_checks = pd.Series(
    {
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_product_region_month_keys": int(
            df.duplicated(KEY_COLS).sum()),
        "missing_product_ids": int(df["product_id"].isna().sum()),
        "missing_region_ids": int(df["region_id"].isna().sum()),
        "missing_months": int(df["year_month"].isna().sum()),
        "labeled_rows": int(df[TARGET_COL].notna().sum()),
        "scoring_rows": int(df[TARGET_COL].isna().sum()),
        "start_month": df["year_month"].min(),
        "end_month": df["year_month"].max()},
    name="value")
display(input_checks.to_frame())

assert input_checks["duplicate_product_region_month_keys"] == 0
assert input_checks[[
    "missing_product_ids", "missing_region_ids", "missing_months"
]].eq(0).all()


,value
rows,120000
columns,131
duplicate_product_region_month_keys,0
missing_product_ids,0
missing_region_ids,0
missing_months,0
labeled_rows,118000
scoring_rows,2000
start_month,2021-01-01 00:00:00
end_month,2025-12-01 00:00:00


# 3. Calculation helpers

`safe_divide` returns missing values instead of infinite values when the denominator is zero. `slope` measures the direction and speed of a short time trend.


In [5]:
def safe_divide(
    numerator: pd.Series, denominator: pd.Series
) -> pd.Series:
    return numerator / denominator.replace(0, np.nan)

def slope(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    if len(values) < 2 or np.all(np.isnan(values)):
        return np.nan
    return float(np.polyfit(np.arange(len(values)), values, 1)[0])


# 4. Product lifecycle and profitability features

Lifecycle features describe product maturity and strategic stage. Profitability features compare observed margin with both an estimated cost-based margin and the product's target margin.

In [6]:
baseline_columns = list(df.columns)
df = df.sort_values(KEY_COLS).reset_index(drop=True)

# Compatibility alias used by the original feature-engineering notebook.
df["product_group"] = df["product_category"]

df["product_age_months"] = (
    (df["year_month"].dt.year - df["effective_launch_date"].dt.year) * 12
    + df["year_month"].dt.month
    - df["effective_launch_date"].dt.month
).clip(lower=0)

lifecycle_order = {"New": 0, "Growth": 1, "Mature": 2, "Decline": 3}
df["lifecycle_stage_num"] = (
    df["lifecycle_stage"].map(lifecycle_order).fillna(-1).astype(int))
df["is_decline_stage"] = df["lifecycle_stage"].eq("Decline").astype(int)
df["is_growth_stage"] = df["lifecycle_stage"].eq("Growth").astype(int)
df["is_mature_product"] = df["product_age_months"].ge(36).astype(int)

df["estimated_gross_profit"] = (
    df["revenue"] - df["units_sold"] * df["base_unit_cost_eur"])
df["estimated_gross_margin_pct"] = safe_divide(
    df["estimated_gross_profit"], df["revenue"])
df["margin_gap_vs_target"] = (
    df["estimated_gross_margin_pct"] - df["target_margin_pct"])
df["actual_margin_gap_vs_target"] = (
    df["gross_margin_pct"] - df["target_margin_pct"])
df["actual_vs_estimated_margin_gap"] = (
    df["gross_margin_pct"] - df["estimated_gross_margin_pct"])
df["gross_profit_per_unit"] = safe_divide(
    df["gross_profit_eur"], df["units_sold"])
df["cost_variance_to_revenue"] = safe_divide(
    df["cost_variance_eur"], df["revenue"])
df["return_value_rate"] = safe_divide(
    df["return_value_eur"], df["revenue"])

In [7]:
lifecycle_check = df[
    [
        "product_id", "year_month", "effective_launch_date", "lifecycle_stage",
        "product_age_months", "lifecycle_stage_num","gross_margin_pct", "estimated_gross_margin_pct", "target_margin_pct",
        "target_margin_pct", "margin_gap_vs_target"]
].sample(8, random_state=RANDOM_SEED)
display(lifecycle_check)

print(
    "Products with a negative estimated margin gap: "
    f"{df['margin_gap_vs_target'].lt(0).mean():.1%} of rows")

,product_id,year_month,effective_launch_date,lifecycle_stage,product_age_months,lifecycle_stage_num,gross_margin_pct,estimated_gross_margin_pct,target_margin_pct,target_margin_pct,margin_gap_vs_target
71787,1119,2023-04-01,2021-01-03,Mature,27,2,0.336,0.366,0.340,0.340,0.026
67218,1112,2022-07-01,2020-01-01,Mature,30,2,0.401,0.429,0.340,0.340,0.089
54066,1090,2021-07-01,2020-01-01,Decline,18,3,NaN,NaN,0.390,0.390,NaN
7168,1011,2023-05-01,2019-01-01,Mature,52,2,0.256,0.309,0.290,0.290,0.019
29618,1049,2024-03-01,2019-01-01,Decline,62,3,0.130,0.216,0.270,0.270,-0.054
101425,1169,2023-02-01,2020-01-01,Decline,37,3,-0.179,-0.135,0.220,0.220,-0.355
20441,1034,2024-06-01,2021-01-01,Decline,41,3,0.112,0.152,0.240,0.240,-0.088
2662,1004,2022-11-01,2021-01-01,Mature,22,2,0.165,0.189,0.290,0.290,-0.101


Products with a negative estimated margin gap: 38.4% of rows


# 5. Marketing and digital-demand features

Raw activity counts become interpretable funnel and efficiency ratios. Historical campaign baselines use only prior months, while current-month conversion ratios are available at the prediction point.

Missing source rows remain missing rather than being interpreted as zero activity; the preprocessing availability flag remains available for audit purposes.


In [8]:
for column in [
    "campaign_spend_eur", "website_visits", "product_page_views",
    "demo_requests", "marketing_qualified_leads", "campaign_clicks",
    "campaign_impressions",
]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["campaign_spend_per_visit"] = safe_divide(
    df["campaign_spend_eur"], df["website_visits"])
df["campaign_spend_per_demo"] = safe_divide(
    df["campaign_spend_eur"], df["demo_requests"])
df["page_view_to_visit_ratio"] = safe_divide(
    df["product_page_views"], df["website_visits"])
df["demo_conversion_rate"] = safe_divide(
    df["demo_requests"], df["product_page_views"])
df["mql_conversion_rate"] = safe_divide(
    df["marketing_qualified_leads"], df["product_page_views"])
df["click_to_mql_rate"] = safe_divide(
    df["marketing_qualified_leads"], df["campaign_clicks"])
df["digital_interest_per_unit"] = safe_divide(
    df["product_page_views"], df["units_sold"])
df["campaign_active_with_no_sales"] = (
    df["campaign_spend_eur"].gt(0) & df["units_sold"].eq(0)
).astype(int)

product_region_group = df.groupby(
    ["product_id", "region_id"], group_keys=False)
df["campaign_spend_lag_1m"] = product_region_group[
    "campaign_spend_eur"
].shift(1)
df["campaign_spend_rolling_3m"] = product_region_group[
    "campaign_spend_eur"
].transform(
    lambda series: series.shift(1).rolling(3, min_periods=1).sum())

website_visits_lag_1m = product_region_group["website_visits"].shift(1)
demo_requests_lag_1m = product_region_group["demo_requests"].shift(1)
df["website_visits_growth_pct"] = safe_divide(
    df["website_visits"] - website_visits_lag_1m,
    website_visits_lag_1m)
df["demo_requests_growth_pct"] = safe_divide(
    df["demo_requests"] - demo_requests_lag_1m,
    demo_requests_lag_1m)


In [9]:
marketing_check = df[
    [
        "product_id", "region", "year_month", "campaign_spend_eur",
        "website_visits", "product_page_views", "demo_requests",
        "campaign_spend_per_visit", "demo_conversion_rate",
        "campaign_spend_lag_1m", "campaign_spend_rolling_3m"]
].loc[df["market_activity_available_flag"].eq(1)].head(10)
display(marketing_check)

,product_id,region,year_month,campaign_spend_eur,website_visits,product_page_views,demo_requests,campaign_spend_per_visit,demo_conversion_rate,campaign_spend_lag_1m,campaign_spend_rolling_3m
0,1000,DACH,2021-01-01,0.000,270,514,1,0.000,0.002,NaN,NaN
1,1000,DACH,2021-02-01,0.000,264,354,2,0.000,0.006,0.000,0.000
2,1000,DACH,2021-03-01,"6,135.830",3137,8698,36,1.956,0.004,0.000,0.000
3,1000,DACH,2021-04-01,0.000,242,432,4,0.000,0.009,"6,135.830","6,135.830"
4,1000,DACH,2021-05-01,"3,101.770",1634,4185,20,1.898,0.005,0.000,"6,135.830"
5,1000,DACH,2021-06-01,0.000,227,581,3,0.000,0.005,"3,101.770","9,237.600"
6,1000,DACH,2021-07-01,0.000,215,385,2,0.000,0.005,0.000,"3,101.770"
7,1000,DACH,2021-08-01,"3,358.130",2329,3280,13,1.442,0.004,0.000,"3,101.770"
8,1000,DACH,2021-09-01,"6,514.810",3759,6975,37,1.733,0.005,"3,358.130","3,358.130"
9,1000,DACH,2021-10-01,0.000,262,577,2,0.000,0.003,"6,514.810","9,872.940"


# 6. Supply-chain and return-pressure features

These features describe whether demand can be supported operationally. The `supply_pressure_score` combines stockouts, zero stock, backorders and the external supply-pressure index on 0–100 scale.


In [10]:
df["production_to_sales_ratio"] = safe_divide(
    df["production_units"], df["units_sold"])
df["ending_stock_to_sales_ratio"] = safe_divide(
    df["ending_stock_units"], df["units_sold"])
df["inventory_months_cover"] = safe_divide(
    df["ending_stock_units"], df["rolling_units_mean_3m"])
df["inventory_turnover_proxy"] = safe_divide(
    df["units_sold"], df["ending_stock_units"])
df["stock_change_units"] = (
    df["ending_stock_units"] - df["opening_stock_units"])
df["backorder_rate"] = safe_divide(
    df["backorder_units"], df["units_sold"])

stockout_component = df["stockout_flag"].fillna(0).clip(0, 1) * 35
zero_stock_component = df["zero_stock_flag"].fillna(0).clip(0, 1) * 25
backorder_component = df["backorder_rate"].fillna(0).clip(0, 1) * 20
external_supply_component = (
    df["supply_pressure_index"].fillna(50).clip(0, 100) / 100 * 20)
df["supply_pressure_score"] = (
    stockout_component
    + zero_stock_component
    + backorder_component
    + external_supply_component
).clip(0, 100)

df["return_pressure_score"] = (
    df["return_rate_units"].fillna(0).clip(0, 1) * 60
    + df["return_value_rate"].fillna(0).clip(0, 1) * 40
).clip(0, 100)
df["stockout_with_demand_flag"] = (
    df["stockout_flag"].eq(1) & df["units_sold"].gt(0)
).astype(int)

In [11]:
supply_check = df[
    [
        "product_id", "region", "year_month", "units_sold",
        "opening_stock_units", "production_units", "ending_stock_units",
        "stockout_flag", "backorder_rate", "supply_pressure_index",
        "supply_pressure_score", "return_pressure_score"]
].loc[df["inventory_available_flag"].eq(1)].head(10)
display(supply_check)


,product_id,region,year_month,units_sold,opening_stock_units,production_units,ending_stock_units,stockout_flag,backorder_rate,supply_pressure_index,supply_pressure_score,return_pressure_score
0,1000,DACH,2021-01-01,81.000,53.000,79.000,51.000,0,0.000,41.530,8.306,0.000
2,1000,DACH,2021-03-01,16.000,23.000,18.000,25.000,0,0.000,45.280,9.056,0.000
3,1000,DACH,2021-04-01,35.000,47.000,28.000,40.000,0,0.000,40.550,8.110,0.000
5,1000,DACH,2021-06-01,160.000,213.000,193.000,246.000,0,0.000,40.190,8.038,0.000
6,1000,DACH,2021-07-01,9.000,10.000,10.000,11.000,0,0.000,40.790,8.158,0.000
9,1000,DACH,2021-10-01,125.000,110.000,102.000,87.000,0,0.000,41.940,8.388,8.142
10,1000,DACH,2021-11-01,75.000,96.000,93.000,114.000,0,0.000,40.680,8.136,0.000
11,1000,DACH,2021-12-01,67.000,65.000,84.000,82.000,0,0.000,40.170,8.034,0.000
13,1000,DACH,2022-02-01,81.000,67.000,64.000,50.000,0,0.000,40.900,8.180,0.000
14,1000,DACH,2022-03-01,253.000,290.000,319.000,356.000,0,0.000,42.170,8.434,0.000


# 7. Market-context features

The market table is repeated across individual products belonging to the same family/group. Percentage changes must therefore be calculated on a **deduplicated market grain**, then joined back to products. Calculating `pct_change()` directly across repeated product rows would compare products within the same month and produce incorrect results.


In [12]:
df["seasonality_adjusted_units"] = safe_divide(
    df["units_sold"], df["seasonality_index"] / 100)
df["demand_competition_ratio"] = safe_divide(
    df["market_demand_index"], df["competitor_pressure_index"])
df["market_tailwind_score"] = (
    (df["market_demand_index"].fillna(100) - 100) * 0.35
    + (100 - df["competitor_pressure_index"].fillna(50)) * 0.20
    + (df["macro_business_index"].fillna(100) - 100) * 0.15
    + (df["market_opportunity_score"].fillna(50) - 50) * 0.20
    + df["market_growth_pct"].fillna(0).clip(-1, 1) * 10)
df["high_competition_flag"] = np.where(
    df["competitor_pressure_index"].notna(),
    df["competitor_pressure_index"].ge(70).astype(int),
    np.nan)
df["weak_market_flag"] = np.where(
    df["market_demand_index"].notna(),
    df["market_demand_index"].lt(90).astype(int),
    np.nan)

market_grain_columns = [
    "product_line", "product_category", "region_id", "year_month"]
market_value_columns = [
    "market_demand_index", "macro_business_index",
    "competitor_pressure_index", "market_opportunity_score",
    "supply_pressure_index"]
unique_market = (
    df[market_grain_columns + market_value_columns]
    .drop_duplicates(market_grain_columns)
    .sort_values(market_grain_columns)
    .reset_index(drop=True))
market_history = unique_market.groupby(
    ["product_line", "product_category", "region_id"],
    group_keys=False)

for source_column, output_column in [
    ("market_demand_index", "market_demand_change_pct"),
    ("macro_business_index", "macro_business_index_pct"),
    ("competitor_pressure_index", "competitor_pressure_change_pct"),
    ("market_opportunity_score", "market_opportunity_change_pct"),
    ("supply_pressure_index", "supply_pressure_change_pct"),
]:
    lagged = market_history[source_column].shift(1)
    unique_market[output_column] = safe_divide(
        unique_market[source_column] - lagged, lagged)

market_change_columns = [
    "market_demand_change_pct", "macro_business_index_pct",
    "competitor_pressure_change_pct", "market_opportunity_change_pct",
    "supply_pressure_change_pct"]
rows_before_merge = len(df)
df = df.merge(
    unique_market[market_grain_columns + market_change_columns],
    on=market_grain_columns,
    how="left",
    validate="m:1")
assert len(df) == rows_before_merge


In [13]:
market_change_check = unique_market[
    market_grain_columns
    + [
        "market_demand_index", "market_demand_change_pct",
        "competitor_pressure_index", "competitor_pressure_change_pct"]
].dropna(subset=["market_demand_index"]).head(12)
display(market_change_check)

,product_line,product_category,region_id,year_month,market_demand_index,market_demand_change_pct,competitor_pressure_index,competitor_pressure_change_pct
0,Accessories,Accessory Kit,1,2021-01-01,102.070,NaN,71.060,NaN
1,Accessories,Accessory Kit,1,2021-02-01,98.970,-0.030,61.980,-0.128
2,Accessories,Accessory Kit,1,2021-03-01,99.520,0.006,62.130,0.002
3,Accessories,Accessory Kit,1,2021-04-01,111.100,0.116,64.880,0.044
4,Accessories,Accessory Kit,1,2021-05-01,102.280,-0.079,65.650,0.012
5,Accessories,Accessory Kit,1,2021-06-01,107.340,0.049,66.190,0.008
6,Accessories,Accessory Kit,1,2021-07-01,84.350,-0.214,59.600,-0.100
7,Accessories,Accessory Kit,1,2021-08-01,86.750,0.028,55.980,-0.061
8,Accessories,Accessory Kit,1,2021-09-01,103.730,0.196,51.460,-0.081
9,Accessories,Accessory Kit,1,2021-10-01,135.960,0.311,50.570,-0.017


# 8. Category-region peer benchmarks

Peer features show whether a product is gaining or losing relevance relative to products in the same category and region. Current-month peer totals are allowed because all products' current-month sales are known when the monthly prediction is produced.


In [14]:
category_region = (
    df.groupby(
        ["product_category", "region_id", "year_month"], as_index=False)
    .agg(
        category_region_units_2=("units_sold", "sum"),
        category_region_revenue=("revenue", "sum"),
        category_region_customers=("unique_customers", "sum"),
        category_region_avg_discount=("avg_discount_pct", "mean"),
        category_region_product_count=("product_id", "nunique")))

rows_before_merge = len(df)
df = df.merge(
    category_region,
    on=["product_category", "region_id", "year_month"],
    how="left",
    validate="m:1")
assert len(df) == rows_before_merge

df["product_unit_share_category_region"] = safe_divide(
    df["units_sold"], df["category_region_units_2"])
df["product_revenue_share_category_region"] = safe_divide(
    df["revenue"], df["category_region_revenue"])
df["customer_share_category_region"] = safe_divide(
    df["unique_customers"], df["category_region_customers"])
df["discount_vs_category_region"] = (
    df["avg_discount_pct"] - df["category_region_avg_discount"])

df["unit_rank_category_region"] = df.groupby(
    ["product_category", "region_id", "year_month"]
)["units_sold"].rank(method="dense", ascending=False)
df["revenue_percentile_category_region"] = df.groupby(
    ["product_category", "region_id", "year_month"]
)["revenue"].rank(method="average", pct=True)


In [15]:
df = df.sort_values(KEY_COLS).reset_index(drop=True)
product_region_group = df.groupby(
    ["product_id", "region_id"], group_keys=False)

df["unit_share_lag_1m"] = product_region_group[
    "product_unit_share_category_region"
].shift(1)
df["unit_share_change_pct"] = safe_divide(
    df["product_unit_share_category_region"] - df["unit_share_lag_1m"],
    df["unit_share_lag_1m"])
df["units_acceleration_3m"] = product_region_group[
    "trend_slope_3m"
].diff()
df["units_vs_6m_avg_pct"] = safe_divide(
    df["units_sold"] - df["rolling_units_mean_6m"],
    df["rolling_units_mean_6m"])
df["revenue_vs_lag_1m_pct"] = safe_divide(
    df["revenue"] - df["revenue_lag_1m"], df["revenue_lag_1m"])

prior_units_mean_6m = product_region_group["units_sold"].transform(
    lambda series: series.shift(1).rolling(6, min_periods=3).mean())
prior_units_std_6m = product_region_group["units_sold"].transform(
    lambda series: series.shift(1).rolling(6, min_periods=3).std())
df["rolling_units_cv_6m"] = safe_divide(
    prior_units_std_6m, prior_units_mean_6m)

df["rolling_revenue_mean_3m"] = product_region_group[
    "revenue"
].transform(
    lambda series: series.shift(1).rolling(3, min_periods=2).mean())
df["rolling_revenue_std_3m"] = product_region_group[
    "revenue"
].transform(
    lambda series: series.shift(1).rolling(3, min_periods=2).std())
df["revenue_volatility_3m"] = safe_divide(
    df["rolling_revenue_std_3m"], df["rolling_revenue_mean_3m"])


In [16]:
peer_check = df[
    [
        "product_id", "product_category", "region", "year_month",
        "units_sold", "category_region_units_2",
        "product_unit_share_category_region", "unit_share_lag_1m",
        "unit_share_change_pct", "unit_rank_category_region"]
].head(12)
display(peer_check)

share_reconciliation = (
    df.groupby(["product_category", "region_id", "year_month"])[
        "product_unit_share_category_region"]
    .sum(min_count=1)
    .dropna())
print(
    "Maximum absolute unit-share reconciliation difference: "
    f"{(share_reconciliation - 1).abs().max():.8f}")


,product_id,product_category,region,year_month,units_sold,category_region_units_2,product_unit_share_category_region,unit_share_lag_1m,unit_share_change_pct,unit_rank_category_region
0,1000,Laptop Pro,DACH,2021-01-01,81.000,"1,513.000",0.054,NaN,NaN,8.000
1,1000,Laptop Pro,DACH,2021-02-01,0.000,"1,127.000",0.000,0.054,-1.000,14.000
2,1000,Laptop Pro,DACH,2021-03-01,16.000,"1,684.000",0.010,0.000,NaN,16.000
3,1000,Laptop Pro,DACH,2021-04-01,35.000,"2,168.000",0.016,0.010,0.699,15.000
4,1000,Laptop Pro,DACH,2021-05-01,0.000,"1,086.000",0.000,0.016,-1.000,14.000
5,1000,Laptop Pro,DACH,2021-06-01,160.000,"1,653.000",0.097,0.000,NaN,4.000
6,1000,Laptop Pro,DACH,2021-07-01,9.000,"1,607.000",0.006,0.097,-0.942,16.000
7,1000,Laptop Pro,DACH,2021-08-01,0.000,"1,809.000",0.000,0.006,-1.000,19.000
8,1000,Laptop Pro,DACH,2021-09-01,0.000,"1,434.000",0.000,0.000,NaN,17.000
9,1000,Laptop Pro,DACH,2021-10-01,125.000,"2,222.000",0.056,0.000,NaN,7.000


Maximum absolute unit-share reconciliation difference: 0.00000000


# 9. Customer-mix features

`customer_product_monthly` contains one customer-product-month observation with segment and churn-risk bands. It supports customer composition without reopening transaction-level data.

Shares are calculated from active customers, while average order units and revenue use total orders as the denominator.


In [17]:
customer_mix_source = customer_product_monthly.copy()
customer_mix_source["is_distributor"] = (
    customer_mix_source["customer_segment"].eq("Distributor").astype(int))
customer_mix_source["is_churn_risk_customer"] = (
    customer_mix_source["churn_risk_band"]
    .astype("string")
    .str.contains("High", case=False, na=False)
    .astype(int))

customer_mix = (
    customer_mix_source.groupby(KEY_COLS, as_index=False)
    .agg(
        distributor_order_share=("is_distributor", "mean"),
        churn_risk_customer_share=("is_churn_risk_customer", "mean"),
        customer_mix_units=("units_sold", "sum"),
        customer_mix_revenue=("revenue", "sum"),
        customer_mix_orders=("order_count", "sum")))
customer_mix["avg_order_units"] = safe_divide(
    customer_mix["customer_mix_units"], customer_mix["customer_mix_orders"])
customer_mix["avg_order_revenue"] = safe_divide(
    customer_mix["customer_mix_revenue"],
    customer_mix["customer_mix_orders"])
customer_mix = customer_mix.drop(
    columns=[
        "customer_mix_units", "customer_mix_revenue", "customer_mix_orders"])

rows_before_merge = len(df)
df = df.merge(customer_mix, on=KEY_COLS, how="left", validate="1:1")
assert len(df) == rows_before_merge

df["avg_customer_churn_probability"] = df[
    "avg_base_churn_probability"]

df = df.sort_values(KEY_COLS).reset_index(drop=True)
product_region_group = df.groupby(
    ["product_id", "region_id"], group_keys=False)
df["avg_customer_size_score_lag_1m"] = product_region_group[
    "avg_customer_size_score"
].shift(1)
df["avg_customer_churn_probability_lag_1m"] = product_region_group[
    "avg_customer_churn_probability"
].shift(1)
df["distributor_order_share_lag_1m"] = product_region_group[
    "distributor_order_share"
].shift(1)
df["churn_risk_customer_share_lag_1m"] = product_region_group[
    "churn_risk_customer_share"
].shift(1)


In [18]:
customer_mix_check = df[
    [
        "product_id", "region", "year_month", "unique_customers",
        "avg_customer_size_score", "distributor_order_share",
        "avg_customer_churn_probability", "churn_risk_customer_share",
        "avg_order_units", "avg_order_revenue"]
].loc[df["unique_customers"].gt(0)].head(10)
display(customer_mix_check)

,product_id,region,year_month,unique_customers,avg_customer_size_score,distributor_order_share,avg_customer_churn_probability,churn_risk_customer_share,avg_order_units,avg_order_revenue
0,1000,DACH,2021-01-01,2.000,6.149,0.000,0.045,0.000,40.500,"28,179.010"
2,1000,DACH,2021-03-01,1.000,7.370,0.000,0.083,0.000,16.000,"11,516.520"
3,1000,DACH,2021-04-01,2.000,7.302,0.000,0.079,0.000,17.500,"11,473.160"
5,1000,DACH,2021-06-01,2.000,8.924,0.000,0.029,0.000,80.000,"48,934.190"
6,1000,DACH,2021-07-01,1.000,2.315,0.000,0.070,0.000,9.000,"6,369.560"
9,1000,DACH,2021-10-01,3.000,14.024,0.000,0.037,0.000,41.667,"28,024.160"
10,1000,DACH,2021-11-01,3.000,8.962,0.333,0.056,0.000,25.000,"17,138.823"
11,1000,DACH,2021-12-01,2.000,33.249,0.000,0.035,0.000,33.500,"24,110.945"
13,1000,DACH,2022-02-01,2.000,30.971,0.500,0.059,0.000,40.500,"26,703.965"
14,1000,DACH,2022-03-01,4.000,7.834,0.250,0.079,0.000,50.600,"30,351.838"


# 10. Momentum and interaction features

Interaction features combine different business perspectives. They help tree-based models recognize patterns such as strong demand with insufficient supply or heavy marketing activity without commercial conversion.


In [19]:
product_region_group = df.groupby(
    ["product_id", "region_id"], group_keys=False)

gross_profit_lag_1m = product_region_group["gross_profit_eur"].shift(1)
pipeline_lag_1m = product_region_group["weighted_pipeline_eur"].shift(1)
return_rate_lag_1m = product_region_group["return_rate_units"].shift(1)

df["gross_profit_growth_pct"] = safe_divide(
    df["gross_profit_eur"] - gross_profit_lag_1m,
    gross_profit_lag_1m)
df["weighted_pipeline_growth_pct"] = safe_divide(
    df["weighted_pipeline_eur"] - pipeline_lag_1m,
    pipeline_lag_1m)
df["return_rate_change_pct"] = safe_divide(
    df["return_rate_units"] - return_rate_lag_1m,
    return_rate_lag_1m)

df["demand_supply_gap"] = (
    df["market_demand_index"] - df["supply_pressure_index"])
df["demand_supply_mismatch_flag"] = np.where(
    df["market_demand_index"].notna()
    & df["supply_pressure_index"].notna(),
    (
        df["market_demand_index"].ge(110)
        & df["supply_pressure_index"].ge(65)
    ).astype(int),
    np.nan)
df["engagement_to_pipeline_ratio"] = safe_divide(
    df["marketing_qualified_leads"], df["pipeline_opportunities"])
df["pipeline_value_per_customer"] = safe_divide(
    df["weighted_pipeline_eur"], df["unique_customers"])
df["marketing_efficiency_score"] = safe_divide(
    df["marketing_qualified_leads"] + df["demo_requests"],
    df["campaign_spend_eur"])

df["commercial_pressure_score"] = (
    df["risk_factor_sales_drop"].fillna(0) * 20
    + df["risk_factor_customer_drop"].fillna(0) * 15
    + df["risk_factor_under_trend"].fillna(0) * 15
    + df["high_competition_flag"].fillna(0) * 15
    + df["weak_market_flag"].fillna(0) * 15
    + df["supply_pressure_score"].fillna(0) * 0.20
).clip(0, 100)
df["revenue_at_risk_proxy"] = (
    df["revenue"] * df["commercial_pressure_score"] / 100)

df = df.replace([np.inf, -np.inf], np.nan)


In [20]:
interaction_check = df[
    [
        "product_id", "region", "year_month", "market_demand_index",
        "supply_pressure_index", "demand_supply_gap",
        "demand_supply_mismatch_flag", "commercial_pressure_score",
        "revenue", "revenue_at_risk_proxy"]
].sample(10, random_state=RANDOM_SEED)
display(interaction_check)


,product_id,region,year_month,market_demand_index,supply_pressure_index,demand_supply_gap,demand_supply_mismatch_flag,commercial_pressure_score,revenue,revenue_at_risk_proxy
71787,1119,North America,2023-04-01,130.310,40.970,89.340,0.000,31.639,"49,361.500","15,617.386"
67218,1112,DACH,2022-07-01,98.420,40.530,57.890,0.000,31.621,"41,586.080","13,150.018"
54066,1090,APAC,2021-07-01,89.420,41.220,48.200,0.000,16.649,0.000,0.000
7168,1011,Eastern Europe,2023-05-01,152.740,41.140,111.600,0.000,1.646,"22,773.570",374.762
29618,1049,Southern Europe,2024-03-01,169.530,40.750,128.780,0.000,1.630,"5,375.650",87.623
101425,1169,DACH,2023-02-01,110.970,40.780,70.190,0.000,1.631,"5,235.130",85.395
20441,1034,DACH,2024-06-01,117.110,40.560,76.550,0.000,1.622,"58,531.770",949.619
2662,1004,Southern Europe,2022-11-01,137.770,39.860,97.910,0.000,16.594,"167,284.520","27,759.862"
20371,1033,Eastern Europe,2023-08-01,108.100,41.090,67.010,0.000,31.644,0.000,0.000
108151,1180,Western Europe,2023-08-01,102.600,41.380,61.220,0.000,31.655,0.000,0.000


# 11. Compatibility and leakage checks

In [21]:
ENGINEERED_COLUMNS = [
    "product_group", "product_age_months", "is_new_product",
    "lifecycle_stage_num", "is_decline_stage", "is_growth_stage",
    "estimated_gross_profit", "estimated_gross_margin_pct",
    "margin_gap_vs_target", "campaign_spend_per_visit",
    "campaign_spend_per_demo", "page_view_to_visit_ratio",
    "demo_conversion_rate", "digital_interest_per_unit",
    "campaign_active_with_no_sales", "campaign_spend_lag_1m",
    "campaign_spend_rolling_3m", "website_visits_growth_pct",
    "demo_requests_growth_pct", "seasonality_adjusted_units",
    "demand_competition_ratio", "market_tailwind_score",
    "high_competition_flag", "weak_market_flag",
    "market_demand_change_pct", "macro_business_index_pct",
    "competitor_pressure_change_pct", "category_region_units_2",
    "category_region_revenue", "category_region_customers",
    "category_region_avg_discount",
    "product_unit_share_category_region",
    "product_revenue_share_category_region",
    "customer_share_category_region", "discount_vs_category_region",
    "unit_share_lag_1m", "unit_share_change_pct",
    "units_acceleration_3m", "units_vs_6m_avg_pct",
    "revenue_vs_lag_1m_pct", "rolling_units_cv_6m",
    "rolling_revenue_mean_3m", "rolling_revenue_std_3m",
    "revenue_volatility_3m", "avg_customer_size_score",
    "distributor_order_share", "avg_customer_churn_probability",
    "churn_risk_customer_share", "avg_order_units",
    "avg_order_revenue", "avg_customer_size_score_lag_1m",
    "avg_customer_churn_probability_lag_1m",
    "distributor_order_share_lag_1m",
    "churn_risk_customer_share_lag_1m"]

missing_features = [
    column for column in ENGINEERED_COLUMNS if column not in df.columns]

compatibility_check = pd.DataFrame(
    {
        "required_feature_count": [
            len(ENGINEERED_COLUMNS)
        ],
        "available_feature_count": [
            len(ENGINEERED_COLUMNS) - len(missing_features)
        ],
        "missing_features": [
            ", ".join(missing_features) or "None"]})
display(compatibility_check)
assert not missing_features


,required_feature_count,available_feature_count,missing_features
0,54,54,None


## 11.1 Model-feature selection
identifiers, audit fields, date keys, targets and future-like columns are excluded in one consolidated set.

In [22]:
CATEGORICAL_FEATURES = [
    column
    for column in [
        "product_line", "product_category", "region",
        "lifecycle_stage", "dominant_margin_category",
        "dominant_order_size_category", "dominant_rep_seniority",
        "campaign_channel"]
    if column in df.columns]

IDENTIFIER_COLUMNS = {
    "product_id", "region_id", "year_month", "target_month",
    "product_name", "sku", "country", "currency"}
AUDIT_DATE_COLUMNS = {
    "launch_date", "effective_launch_date", "observed_first_sale_date",
    "MonthStartDate", "MonthEndDate", "YearMonthKey"}
DATA_QUALITY_AUDIT_COLUMNS = {
    "missing_sales_rep_count", "missing_discount_count",
    "missing_margin_count", "market_activity_available_flag",
    "market_signal_available_flag", "inventory_available_flag"}
TARGET_AND_FUTURE_COLUMNS = {
    TARGET_COL,
    *[
        column
        for column in df.columns
        if column.startswith("future_")
        or ("next_month" in column and column != TARGET_COL)]}

EXCLUDED_FROM_MODEL = (
    IDENTIFIER_COLUMNS
    | AUDIT_DATE_COLUMNS
    | DATA_QUALITY_AUDIT_COLUMNS
    | TARGET_AND_FUTURE_COLUMNS)

NUMERIC_FEATURES = [
    column
    for column in df.columns
    if column not in EXCLUDED_FROM_MODEL
    and column not in CATEGORICAL_FEATURES
    and pd.api.types.is_numeric_dtype(df[column])]

leakage_like_features = [
    column
    for column in NUMERIC_FEATURES + CATEGORICAL_FEATURES
    if column in TARGET_AND_FUTURE_COLUMNS
    or column in IDENTIFIER_COLUMNS]

feature_selection_summary = pd.Series(
    {
        "numeric_features": len(NUMERIC_FEATURES),
        "categorical_features": len(CATEGORICAL_FEATURES),
        "excluded_columns": len(EXCLUDED_FROM_MODEL),
        "leakage_or_id_features_selected": len(leakage_like_features)},
    name="value")
display(feature_selection_summary.to_frame())
print("Leakage/ID columns selected:", leakage_like_features or "None")

assert not leakage_like_features


,value
numeric_features,181
categorical_features,8
excluded_columns,21
leakage_or_id_features_selected,0


Leakage/ID columns selected: None


# 12. Create the feature catalog

The catalog documents each model feature's family, data type, missing rate, timing and leakage status. -> for GitHub , model governance and Power BI documentation.


In [23]:
engineered_columns = [
    column for column in df.columns if column not in baseline_columns]

feature_family_rules = {
    "lifecycle & profitability": [
        "lifecycle", "product_age", "margin", "gross_profit", "cost_"],
    "marketing & digital": [
        "campaign", "website", "page_view", "demo_", "mql_",
        "digital_", "click_",],
    "supply & returns": [
        "inventory", "stock", "supply", "backorder", "return_",
        "production_"],
    "market context": [
        "market_", "competitor", "seasonality", "macro_", "demand_"],
    "peer benchmark": [
        "category_region", "product_unit_share", "product_revenue_share",
        "customer_share", "unit_share", "unit_rank", "revenue_percentile"],
    "customer mix": [
        "customer_size", "customer_churn", "distributor_",
        "churn_risk_customer", "avg_order_"],
    "momentum & stability": [
        "growth", "acceleration", "rolling_", "volatility",
        "vs_lag", "vs_6m"],
    "interaction & risk": [
        "pressure_score", "commercial_pressure", "revenue_at_risk",
        "engagement_to_pipeline", "pipeline_value_per_customer",
        "efficiency_score", "mismatch"]}

prior_only_tokens = ["lag_", "rolling_", "change_pct", "growth_pct"]

catalog_rows = []
for column in NUMERIC_FEATURES + CATEGORICAL_FEATURES:
    feature_family = "existing preprocessed feature"
    for family, tokens in feature_family_rules.items():
        if any(token in column for token in tokens):
            feature_family = family
            break

    if any(token in column for token in prior_only_tokens):
        timing = "current vs prior / prior-only history"
    elif column in [
        "product_line", "product_category", "region", "lifecycle_stage"
    ]:
        timing = "static/current dimension"
    else:
        timing = "current month available at prediction time"

    if column in CATEGORICAL_FEATURES:
        feature_type = "categorical"
    elif set(df[column].dropna().unique()).issubset({0, 1}):
        feature_type = "binary"
    else:
        feature_type = "numeric"

    catalog_rows.append(
        {
            "feature": column,
            "feature_family": feature_family,
            "feature_type": feature_type,
            "timing": timing,
            "missing_rate": round(float(df[column].isna().mean()), 4),
            "engineered_in_this_notebook": column in engineered_columns,
            "leakage_status": "ok",
            "included_in_model": True})

feature_catalog = pd.DataFrame(catalog_rows).sort_values(
    ["feature_family", "feature"], ignore_index=True)
display(feature_catalog.head(40))


,feature,feature_family,feature_type,timing,missing_rate,engineered_in_this_notebook,leakage_status,included_in_model
0,avg_customer_churn_probability,customer mix,numeric,current month available at prediction time,0.418,True,ok,True
1,avg_customer_churn_probability_lag_1m,customer mix,numeric,current vs prior / prior-only history,0.430,True,ok,True
2,avg_customer_size_score,customer mix,numeric,current month available at prediction time,0.418,False,ok,True
3,avg_customer_size_score_lag_1m,customer mix,numeric,current vs prior / prior-only history,0.430,True,ok,True
4,avg_order_revenue,customer mix,numeric,current month available at prediction time,0.418,True,ok,True
5,avg_order_units,customer mix,numeric,current month available at prediction time,0.418,True,ok,True
6,distributor_order_share,customer mix,numeric,current month available at prediction time,0.418,True,ok,True
7,distributor_order_share_lag_1m,customer mix,numeric,current vs prior / prior-only history,0.430,True,ok,True
8,Month,existing preprocessed feature,numeric,current month available at prediction time,0.000,False,ok,True
9,Quarter,existing preprocessed feature,numeric,current month available at prediction time,0.000,False,ok,True


# 13. Chronological train, validation, test and scoring sets

Splits use `target_month`, the month whose risk outcome is predicted. This is stricter than splitting only on the observation month because it prevents the final training label period from overlapping the first validation target period.

Rows without a target are retained separately for current scoring.


In [24]:
labeled = df.dropna(subset=[TARGET_COL]).copy()
labeled[TARGET_COL] = labeled[TARGET_COL].astype(int)
scoring = df[df[TARGET_COL].isna()].copy()

target_months = sorted(labeled["target_month"].dropna().unique())
if len(target_months) < 12:
    raise ValueError(
        "At least 12 labeled target months are required for time splits.")

train_month_count = max(1, int(len(target_months) * 0.70))
validation_month_count = max(1, int(len(target_months) * 0.15))

train_months = target_months[:train_month_count]
validation_months = target_months[
    train_month_count : train_month_count + validation_month_count]
test_months = target_months[
    train_month_count + validation_month_count :]

train = labeled[labeled["target_month"].isin(train_months)].copy()
validation = labeled[
    labeled["target_month"].isin(validation_months)
].copy()
test = labeled[labeled["target_month"].isin(test_months)].copy()

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(split),
            "observation_start": split["year_month"].min(),
            "observation_end": split["year_month"].max(),
            "target_start": split["target_month"].min(),
            "target_end": split["target_month"].max(),
        }
        for name, split in [
            ("train", train),
            ("validation", validation),
            ("test", test),
            ("scoring / unlabeled", scoring)]])
display(split_summary)

assert train["target_month"].max() < validation["target_month"].min()
assert validation["target_month"].max() < test["target_month"].min()


,split,rows,observation_start,observation_end,target_start,target_end
0,train,82000,2021-01-01,2024-05-01,2021-02-01,2024-06-01
1,validation,16000,2024-06-01,2025-01-01,2024-07-01,2025-02-01
2,test,20000,2025-02-01,2025-11-01,2025-03-01,2025-12-01
3,scoring / unlabeled,2000,2025-12-01,2025-12-01,2026-01-01,2026-01-01


In [25]:
target_distribution_by_split = pd.concat(
    [
        split[TARGET_COL]
        .value_counts(normalize=True)
        .rename(name)
        for name, split in [
            ("train", train),
            ("validation", validation),
            ("test", test)]],
    axis=1,
).fillna(0)
display(target_distribution_by_split)


,train,validation,test
next_month_risk_label,,,
0,0.672,0.668,0.677
1,0.280,0.292,0.284
2,0.048,0.040,0.039


# 14. Fit preprocessing on training data

Numeric fields use training medians followed by standardization. Categorical fields use training modes and training category levels. 

An explicit `__UNSEEN__` indicator captures categories that appear after the training period.

In [26]:
def fit_pandas_preprocessor(
    train_data: pd.DataFrame,
    numeric_features: list[str],
    categorical_features: list[str],
) -> dict[str, object]:
    numeric_medians = train_data[numeric_features].median(
        numeric_only=True
    ).fillna(0)
    filled_numeric = train_data[numeric_features].fillna(numeric_medians)
    numeric_means = filled_numeric.mean()
    numeric_stds = filled_numeric.std().replace(0, 1).fillna(1)

    categorical_modes = {}
    categorical_levels = {}
    for column in categorical_features:
        mode = train_data[column].mode(dropna=True)
        categorical_modes[column] = (
            mode.iloc[0] if len(mode) else "Unknown")
        values = (
            train_data[column]
            .fillna(categorical_modes[column])
            .astype(str))
        categorical_levels[column] = sorted(values.unique().tolist())

    return {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "numeric_medians": numeric_medians,
        "numeric_means": numeric_means,
        "numeric_stds": numeric_stds,
        "categorical_modes": categorical_modes,
        "categorical_levels": categorical_levels}

preprocessor = fit_pandas_preprocessor(
    train, NUMERIC_FEATURES, CATEGORICAL_FEATURES)

fitted_numeric_example = pd.DataFrame(
    {
        "median": preprocessor["numeric_medians"],
        "mean": preprocessor["numeric_means"],
        "std": preprocessor["numeric_stds"]}
).head(15)
fitted_categorical_example = pd.DataFrame(
    {
        "feature": CATEGORICAL_FEATURES,
        "training_mode": [
            preprocessor["categorical_modes"][column]
            for column in CATEGORICAL_FEATURES],
        "training_levels": [
            len(preprocessor["categorical_levels"][column])
            for column in CATEGORICAL_FEATURES]})
display(fitted_numeric_example)
display(fitted_categorical_example)


,median,mean,std
units_sold,11.000,37.631,64.497
revenue,"2,260.900","18,205.758","42,546.434"
unique_customers,1.000,0.940,1.094
avg_discount_pct,0.120,0.122,0.041
units_lag_1m,11.000,36.882,63.885
units_lag_3m,11.000,35.389,62.335
revenue_lag_1m,"2,235.770","17,753.660","42,116.800"
rolling_units_mean_3m,27.000,36.892,40.163
rolling_units_mean_6m,29.167,36.609,31.462
rolling_units_std_3m,29.670,38.373,45.813


,feature,training_mode,training_levels
0,product_line,IT Devices,7
1,product_category,Accessory Kit,10
2,region,North America,8
3,lifecycle_stage,Mature,4
4,dominant_margin_category,Low Margin,3
5,dominant_order_size_category,Medium Order,3
6,dominant_rep_seniority,Professional,5
7,campaign_channel,No Active Campaign,7


In [27]:
def transform_with_pandas_preprocessor(
    dataframe: pd.DataFrame,
    fitted_preprocessor: dict[str, object],
) -> pd.DataFrame:
    numeric_features = fitted_preprocessor["numeric_features"]
    categorical_features = fitted_preprocessor["categorical_features"]

    numeric_data = dataframe[numeric_features].copy()
    numeric_data = numeric_data.fillna(
        fitted_preprocessor["numeric_medians"])
    numeric_data = (
        numeric_data - fitted_preprocessor["numeric_means"]
    ) / fitted_preprocessor["numeric_stds"]

    encoded_parts = [numeric_data.reset_index(drop=True)]

    for column in categorical_features:
        values = (
            dataframe[column]
            .fillna(fitted_preprocessor["categorical_modes"][column])
            .astype(str))
        levels = fitted_preprocessor["categorical_levels"][column]
        encoded = pd.DataFrame(
            {
                f"{column}__{level}": (values == level)
                .astype(int)
                .to_numpy()
                for level in levels})
        encoded[f"{column}____UNSEEN__"] = (~values.isin(levels)).astype(
            int
        ).to_numpy()
        encoded_parts.append(encoded)

    return pd.concat(encoded_parts, axis=1)

train_ml = transform_with_pandas_preprocessor(train, preprocessor)
validation_ml = transform_with_pandas_preprocessor(validation, preprocessor)
test_ml = transform_with_pandas_preprocessor(test, preprocessor)
scoring_ml = transform_with_pandas_preprocessor(scoring, preprocessor)

feature_names = train_ml.columns.tolist()
assert feature_names == validation_ml.columns.tolist()
assert feature_names == test_ml.columns.tolist()
assert feature_names == scoring_ml.columns.tolist()

train_ml[TARGET_COL] = train[TARGET_COL].to_numpy()
validation_ml[TARGET_COL] = validation[TARGET_COL].to_numpy()
test_ml[TARGET_COL] = test[TARGET_COL].to_numpy()

transformed_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test", "scoring"],
        "rows": [
            len(train_ml), len(validation_ml), len(test_ml), len(scoring_ml)],
        "feature_columns": [len(feature_names)] * 4,
        "missing_feature_cells": [
            int(matrix[feature_names].isna().sum().sum())
            for matrix in [
                train_ml, validation_ml, test_ml, scoring_ml]]})
display(transformed_summary)
display(train_ml.head(3))


,split,rows,feature_columns,missing_feature_cells
0,train,82000,236,0
1,validation,16000,236,0
2,test,20000,236,0
3,scoring,2000,236,0


,units_sold,revenue,unique_customers,avg_discount_pct,units_lag_1m,units_lag_3m,revenue_lag_1m,rolling_units_mean_3m,rolling_units_mean_6m,rolling_units_std_3m,volatility_3m,mom_units_growth_pct,customer_count_growth_pct,trend_slope_3m,trend_slope_6m,stockout_flag,backorder_units,market_demand_index,competitor_pressure_index,campaign_flag,website_visits,demo_requests,risk_factor_sales_drop,risk_factor_customer_drop,risk_factor_volatility,risk_factor_under_trend,is_declining_product,is_new_product,base_list_price_eur,base_unit_cost_eur,target_margin_pct,product_growth_factor,fx_to_eur,market_growth_factor,margin_factor,order_count,active_sales_reps,avg_asp_eur,gross_profit_eur,cogs_eur,discount_value_eur,outlier_order_count,gross_margin_pct,avg_customer_size_score,avg_base_churn_probability,enterprise_customer_count,high_churn_customer_count,covered_annual_quota_eur,no_sales_flag,opening_stock_units,production_units,ending_stock_units,zero_stock_flag,inventory_value_eur,supply_coverage_ratio,standard_unit_cost_eur,actual_unit_cost_eur,cost_variance_eur,cost_variance_pct,return_units,...,pipeline_value_per_customer,marketing_efficiency_score,commercial_pressure_score,revenue_at_risk_proxy,product_line__Accessories,product_line__Components,product_line__IT Devices,product_line__Industrial Tech,product_line__Legacy Products,product_line__Medical Devices,product_line__Services,product_line____UNSEEN__,product_category__Accessory Kit,product_category__Connectivity Module,product_category__Industrial Scanner,product_category__Laptop Pro,product_category__Laptop Standard,product_category__Legacy Workstation,product_category__Medical Sensor,product_category__Monitoring Device,product_category__Service Contract,product_category__Tablet Enterprise,product_category____UNSEEN__,region__APAC,region__Benelux,region__DACH,region__Eastern Europe,region__North America,region__Northern Europe,region__Southern Europe,region__Western Europe,region____UNSEEN__,lifecycle_stage__Decline,lifecycle_stage__Growth,lifecycle_stage__Mature,lifecycle_stage__New,lifecycle_stage____UNSEEN__,dominant_margin_category__High Margin,dominant_margin_category__Low Margin,dominant_margin_category__Medium Margin,dominant_margin_category____UNSEEN__,dominant_order_size_category__Large Order,dominant_order_size_category__Medium Order,dominant_order_size_category__Small Order,dominant_order_size_category____UNSEEN__,dominant_rep_seniority__Junior,dominant_rep_seniority__Key Account,dominant_rep_seniority__Professional,dominant_rep_seniority__Senior,dominant_rep_seniority__Unknown,dominant_rep_seniority____UNSEEN__,campaign_channel__Content,campaign_channel__Email,campaign_channel__No Active Campaign,campaign_channel__Paid Search,campaign_channel__Partner,campaign_channel__Trade Fair,campaign_channel__Webinar,campaign_channel____UNSEEN__,next_month_risk_label
0,0.672,0.897,0.968,-0.903,-0.405,-0.391,-0.368,-0.246,-0.237,-0.190,-0.016,-0.155,-0.214,-0.005,-0.016,-0.043,-0.017,-0.763,0.872,-0.475,-0.339,-0.417,-0.129,-0.664,-0.533,-0.860,1.709,-0.264,0.609,0.568,-0.334,0.047,0.549,0.223,0.095,0.948,0.974,0.675,0.777,0.886,0.302,-0.110,-0.144,-0.247,-0.900,1.317,-0.330,0.721,-0.875,-0.213,0.243,-0.287,-0.043,0.197,-2.015,0.512,0.589,1.400,1.201,-0.099,...,0.284,-0.054,-0.974,-0.102,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0
1,-0.583,-0.428,-0.859,-0.044,0.691,-0.391,0.917,-0.246,-0.237,-0.190,-0.016,-0.212,-0.944,-0.005,-0.016,-0.043,-0.017,-0.595,1.089,-0.475,-0.344,-0.377,-0.129,1.505,-0.533,-0.860,1.709,-0.264,0.609,0.568,-0.334,0.047,0.549,0.223,0.095,-0.855,-0.856,-0.085,-0.370,-0.423,-0.385,-0.110,-0.063,-0.165,-0.105,-0.503,-0.330,-0.786,1.143,-0.199,-0.183,-0.197,-0.043,-0.223,0.003,0.580,0.551,-0.307,-0.406,-0.099,...,-0.235,-0.054,0.111,-0.315,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0
2,-0.335,-0.157,0.055,-0.915,-0.577,-0.391,-

# 15. Final quality assurance

The final checks protect schema compatibility, feature uniqueness, target handling, chronological splits, leakage safety, transformations and business-value ranges.


In [28]:
final_checks = {
    "input_columns_preserved": set(baseline_columns).issubset(df.columns),
    "engineered_columns_preserved": not missing_features,
    "unique_product_region_month_key": not df.duplicated(KEY_COLS).any(),
    "target_not_selected_as_feature": TARGET_COL not in (
        NUMERIC_FEATURES + CATEGORICAL_FEATURES),
    "ids_not_selected_as_features": not any(
        column in NUMERIC_FEATURES + CATEGORICAL_FEATURES
        for column in IDENTIFIER_COLUMNS),
    "no_future_like_features": not any(
        column.startswith("future_")
        for column in NUMERIC_FEATURES + CATEGORICAL_FEATURES),
    "final_rows_remain_available_for_scoring": len(scoring) > 0,
    "split_order_is_chronological": (
        train["target_month"].max() < validation["target_month"].min()
        and validation["target_month"].max() < test["target_month"].min()),
    "ml_feature_names_match_across_splits": (
        feature_names == validation_ml.columns[:-1].tolist()
        and feature_names == test_ml.columns[:-1].tolist()
        and feature_names == scoring_ml.columns.tolist()),
    "ml_matrices_have_no_missing_features": all(
        not matrix[feature_names].isna().any().any()
        for matrix in [train_ml, validation_ml, test_ml, scoring_ml]),
    "supply_pressure_score_in_range": df[
        "supply_pressure_score"
    ].dropna().between(0, 100).all(),
    "commercial_pressure_score_in_range": df[
        "commercial_pressure_score"
    ].dropna().between(0, 100).all(),
    "feature_catalog_has_one_row_per_feature": (
        feature_catalog["feature"].is_unique
        and len(feature_catalog)
        == len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES))}

final_qa = pd.DataFrame(
    {"check": final_checks.keys(), "passed": final_checks.values()})
display(final_qa)

if not all(final_checks.values()):
    raise AssertionError("At least one final feature-engineering check failed.")

print("All final feature-engineering checks passed.")


,check,passed
0,input_columns_preserved,True
1,engineered_columns_preserved,True
2,unique_product_region_month_key,True
3,target_not_selected_as_feature,True
4,ids_not_selected_as_features,True
5,no_future_like_features,True
6,final_rows_remain_available_for_scoring,True
7,split_order_is_chronological,True
8,ml_feature_names_match_across_splits,True
9,ml_matrices_have_no_missing_features,True


All final feature-engineering checks passed.


# 16. Export 

In [29]:
feature_engineered_export = df.copy()
feature_engineered_export["year_month"] = feature_engineered_export[
    "year_month"
].dt.strftime("%Y-%m")
feature_engineered_export["target_month"] = feature_engineered_export[
    "target_month"
].dt.strftime("%Y-%m")

feature_engineered_export.to_csv(
    OUTPUT_DIR / "feature_engineered_modeling_dataset.csv", index=False)
feature_catalog.to_csv(OUTPUT_DIR / "feature_catalog.csv", index=False)

train_ml.to_csv(
    OUTPUT_DIR / "train_feature_engineered_ml_ready.csv", index=False)
validation_ml.to_csv(
    OUTPUT_DIR / "valid_feature_engineered_ml_ready.csv", index=False)
test_ml.to_csv(
    OUTPUT_DIR / "test_feature_engineered_ml_ready.csv", index=False)
scoring_ml.to_csv(
    OUTPUT_DIR / "scoring_feature_engineered_ml_ready.csv", index=False)

traceability_columns = ["product_id", "year_month", "region_id", "region", "target_month"]

def export_ids(
    dataframe: pd.DataFrame,
    path: Path,
    include_target: bool,
) -> None:
    columns = traceability_columns + ([TARGET_COL] if include_target else [])
    ids = dataframe[columns].copy()
    ids["year_month"] = ids["year_month"].dt.strftime("%Y-%m")
    ids["target_month"] = ids["target_month"].dt.strftime("%Y-%m")
    ids.to_csv(path, index=False)

export_ids(
    train, OUTPUT_DIR / "train_feature_engineered_ids.csv", True)
export_ids(
    validation, OUTPUT_DIR / "valid_feature_engineered_ids.csv", True)
export_ids(test, OUTPUT_DIR / "test_feature_engineered_ids.csv", True)
export_ids(
    scoring, OUTPUT_DIR / "scoring_feature_engineered_ids.csv", False)


In [30]:
preprocessor_parameters = {
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_medians": preprocessor["numeric_medians"].to_dict(),
    "numeric_means": preprocessor["numeric_means"].to_dict(),
    "numeric_stds": preprocessor["numeric_stds"].to_dict(),
    "categorical_modes": preprocessor["categorical_modes"],
    "categorical_levels": preprocessor["categorical_levels"],
    "encoded_feature_names": feature_names}
with open(
    OUTPUT_DIR / "feature_engineering_preprocessor.json",
    "w",
    encoding="utf-8") as file:
    json.dump(preprocessor_parameters, file, indent=2, default=str)

quality_report = {
    "input_shape": list(modeling_input.shape),
    "feature_engineered_dataset_shape": list(df.shape),
    "new_engineered_column_count": len(engineered_columns),
    "numeric_feature_count": len(NUMERIC_FEATURES),
    "categorical_feature_count": len(CATEGORICAL_FEATURES),
    "ml_feature_count_after_encoding": len(feature_names),
    "split_shapes": {
        "train": list(train_ml.shape),
        "validation": list(validation_ml.shape),
        "test": list(test_ml.shape),
        "scoring": list(scoring_ml.shape)},
    "target_distribution": (
        df[TARGET_COL]
        .value_counts(dropna=False)
        .sort_index()
        .astype(int)
        .to_dict()),
    "top_missing_rates": (
        df[NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_COL]]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .head(20)
        .round(4)
        .to_dict()),
    "leakage_like_feature_columns": leakage_like_features,
    "final_checks": {
        key: bool(value) for key, value in final_checks.items()}}
with open(
    OUTPUT_DIR / "feature_engineering_quality_report.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(quality_report, file, indent=2, default=str)


In [31]:
expected_output_files = [
    "feature_engineered_modeling_dataset.csv",
    "feature_catalog.csv",
    "train_feature_engineered_ml_ready.csv",
    "valid_feature_engineered_ml_ready.csv",
    "test_feature_engineered_ml_ready.csv",
    "scoring_feature_engineered_ml_ready.csv",
    "train_feature_engineered_ids.csv",
    "valid_feature_engineered_ids.csv",
    "test_feature_engineered_ids.csv",
    "scoring_feature_engineered_ids.csv",
    "feature_engineering_preprocessor.json",
    "feature_engineering_quality_report.json"]

output_summary = pd.DataFrame(
    [
        {
            "file": filename,
            "exists": (OUTPUT_DIR / filename).exists(),
            "size_bytes": (
                (OUTPUT_DIR / filename).stat().st_size
                if (OUTPUT_DIR / filename).exists()
                else 0)}
        for filename in expected_output_files])
display(output_summary)
assert output_summary["exists"].all()

print("\nFeature engineering complete")
print("-" * 80)
print(f"Feature-engineered dataset: {df.shape}")
print(f"New engineered columns:    {len(engineered_columns)}")
print(f"Selected model features:   {len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)}")
print(f"Encoded ML features:       {len(feature_names)}")
print(f"Output folder:             {OUTPUT_DIR}")


,file,exists,size_bytes
0,feature_engineered_modeling_dataset.csv,True,192697324
1,feature_catalog.csv,True,21501
2,train_feature_engineered_ml_ready.csv,True,303911297
3,valid_feature_engineered_ml_ready.csv,True,59104242
4,test_feature_engineered_ml_ready.csv,True,73898250
5,scoring_feature_engineered_ml_ready.csv,True,7369201
6,train_feature_engineered_ids.csv,True,3157075
7,valid_feature_engineered_ids.csv,True,616075
8,test_feature_engineered_ids.csv,True,770075
9,scoring_feature_engineered_ids.csv,True,73053



Feature engineering complete
--------------------------------------------------------------------------------
Feature-engineered dataset: (120000, 214)
New engineered columns:    83
Selected model features:   189
Encoded ML features:       236
Output folder:             ..\data\full_data\feature_engineering
